# Advanced Problems with Solutions: Dynamic Typing

This notebook contains advanced, solution-based exercises on Python's dynamic typing model.

## Best practices emphasized
- Distinguish clearly between a **name** and the **object** it references.
- Use `type()` and `isinstance()` appropriately.
- Prefer behavior-based design (duck typing) when suitable.
- Use type hints to improve readability and tooling, while remembering they do not enforce runtime types by themselves.
- Avoid writing brittle code that assumes a variable name has a permanent type.

> Note: These exercises are written for Python 3 and are intended to deepen conceptual understanding, not just syntax familiarity.

## Problem 1 — A name is not a type

**Task:** Show that a single variable name can reference objects of different types over time, and explain what this implies about dynamic typing.

### Solution idea
In Python, the variable name does not own a fixed type. The type belongs to the object currently referenced by that name.

In [1]:
# Solution 1
a = "hello"
print(a, type(a))

a = 10
print(a, type(a))

a = [1, 2, 3]
print(a, type(a))

a = lambda x: x ** 2
print(a(5), type(a))

hello <class 'str'>
10 <class 'int'>
[1, 2, 3] <class 'list'>
25 <class 'function'>


### Discussion
- The name `a` is reused several times.
- Each time, `a` points to a different object.
- The object's type changes, but the name itself has no permanent type attached to it.
- This is the essence of dynamic typing in Python.

---

## Problem 2 — Assignment does not copy type information

**Task:** Show that assigning one variable to another does not create a special typed relationship between the names.

**Question:** If `b = a`, does `b` become permanently tied to the same future types as `a`?

**Answer:** No. Both names may initially reference the same object, but later rebinding either name is independent.

In [2]:
# Solution 2
a = [1, 2, 3]
b = a

print("Initially:")
print("a =", a, "type:", type(a))
print("b =", b, "type:", type(b))
print("a is b ->", a is b)

a = "now I am a string"

print("\nAfter rebinding a:")
print("a =", a, "type:", type(a))
print("b =", b, "type:", type(b))
print("a is b ->", a is b)

Initially:
a = [1, 2, 3] type: <class 'list'>
b = [1, 2, 3] type: <class 'list'>
a is b -> True

After rebinding a:
a = now I am a string type: <class 'str'>
b = [1, 2, 3] type: <class 'list'>
a is b -> False


### Discussion
- At first, `a` and `b` reference the same list.
- After rebinding `a`, only `a` changes.
- `b` still references the original list.
- Names are independent labels, not containers with fixed types.

---

## Problem 3 — Mutation vs rebinding

**Task:** Demonstrate the difference between mutating an object and rebinding a variable name.

**Why this matters:** Many mistakes in dynamically typed code come from confusing object mutation with name reassignment.

In [3]:
# Solution 3
x = [1, 2]
y = x

print("Before mutation:")
print("x =", x)
print("y =", y)

x.append(3)

print("\nAfter mutating x:")
print("x =", x)
print("y =", y)

x = "rebound to string"

print("\nAfter rebinding x:")
print("x =", x, "type:", type(x))
print("y =", y, "type:", type(y))

Before mutation:
x = [1, 2]
y = [1, 2]

After mutating x:
x = [1, 2, 3]
y = [1, 2, 3]

After rebinding x:
x = rebound to string type: <class 'str'>
y = [1, 2, 3] type: <class 'list'>


### Discussion
- `x.append(3)` mutates the shared list object, so both `x` and `y` reflect the change.
- `x = "rebound to string"` makes `x` reference a new object.
- `y` continues to point to the old list.
- Dynamic typing becomes much easier to reason about once names and objects are kept separate conceptually.

---

## Problem 4 — `type()` vs `isinstance()`

**Task:** Compare `type(obj) is T` with `isinstance(obj, T)`.

**Question:** Which is generally better when you want to support inheritance?

**Answer:** `isinstance()` is usually better because it respects inheritance relationships.

In [4]:
# Solution 4
class Animal:
    pass

class Dog(Animal):
    pass

d = Dog()

print("type(d) is Dog ->", type(d) is Dog)
print("type(d) is Animal ->", type(d) is Animal)
print("isinstance(d, Dog) ->", isinstance(d, Dog))
print("isinstance(d, Animal) ->", isinstance(d, Animal))

type(d) is Dog -> True
type(d) is Animal -> False
isinstance(d, Dog) -> True
isinstance(d, Animal) -> True


### Discussion
- `type(d) is Animal` is `False` because `d` is not exactly an `Animal`; it is a `Dog`.
- `isinstance(d, Animal)` is `True` because `Dog` inherits from `Animal`.
- In most real code, `isinstance()` is more flexible and better aligned with Python's object model.

---

## Problem 5 — Dynamic typing and duck typing

**Task:** Write a function that works with different object types as long as they support the needed operation.

**Goal:** Show that Python often prefers behavior over explicit type checks.

In [5]:
# Solution 5
def describe_length(obj):
    return f"Length is {len(obj)}"

samples = [
    "dynamic",
    [1, 2, 3, 4],
    (10, 20),
    {"a": 1, "b": 2, "c": 3}
]

for item in samples:
    print(type(item), "->", describe_length(item))

<class 'str'> -> Length is 7
<class 'list'> -> Length is 4
<class 'tuple'> -> Length is 2
<class 'dict'> -> Length is 3


### Discussion
- The function does not care whether the input is a string, list, tuple, or dictionary.
- It only assumes that `len(obj)` is supported.
- This is a classic example of duck typing: if an object behaves as needed, it is acceptable.
- This style is often more extensible than strict manual type checks.

---

## Problem 6 — When dynamic typing causes runtime errors

**Task:** Construct an example where a function works for one type but fails for another, then improve the design.

**Key lesson:** Dynamic typing increases flexibility, but errors may appear at runtime if assumptions about behavior are wrong.

In [6]:
# Solution 6
def square_all(values):
    return [x ** 2 for x in values]

print(square_all([1, 2, 3]))

try:
    print(square_all([1, "a", 3]))
except TypeError as ex:
    print("Runtime error:", ex)

def square_numeric_values(values):
    result = []
    for x in values:
        if not isinstance(x, (int, float, complex)):
            raise TypeError(f"Expected numeric value, got {type(x).__name__}")
        result.append(x ** 2)
    return result

print(square_numeric_values([1, 2.5, 3]))

[1, 4, 9]
Runtime error: unsupported operand type(s) for ** or pow(): 'str' and 'int'
[1, 6.25, 9]


### Discussion
- The first function assumes all elements support exponentiation.
- That assumption is validated only at runtime.
- The improved version makes expectations explicit and fails with a clearer error.
- Best practice depends on context: sometimes behavior-based code is ideal, and sometimes explicit validation is better.

---

## Problem 7 — Type hints do not change runtime typing

**Task:** Show that Python type hints improve readability and tooling, but do not enforce types at runtime by themselves.

**Question:** Does annotating a variable or function parameter make Python statically typed at runtime?

**Answer:** No. Type hints are metadata unless additional tools enforce them.

In [7]:
# Solution 7
def greet(name: str) -> str:
    return "Hello, " + name

print(greet("Python"))

try:
    print(greet(100))
except TypeError as ex:
    print("Runtime error:", ex)

print(greet.__annotations__)

Hello, Python
Runtime error: can only concatenate str (not "int") to str
{'name': <class 'str'>, 'return': <class 'str'>}


### Discussion
- The annotations say `name` should be a `str` and the function returns a `str`.
- Python does not automatically enforce that annotation.
- The failure occurs only because string concatenation with an integer is invalid, not because of the annotation itself.
- Tools such as `mypy`, IDEs, and linters can use annotations to catch issues earlier.

---

## Problem 8 — A function that adapts to multiple input types

**Task:** Write a function that accepts an `int`, `float`, or numeric string and returns a normalized float.

**Goal:** Show how to safely handle dynamic inputs without writing confusing code.

In [8]:
# Solution 8
def to_float(value):
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        return float(value)
    raise TypeError(f"Unsupported type: {type(value).__name__}")

samples = [10, 3.14, "2.718"]
for s in samples:
    print(s, "->", to_float(s))

try:
    print(to_float([1, 2]))
except TypeError as ex:
    print("Runtime error:", ex)

10 -> 10.0
3.14 -> 3.14
2.718 -> 2.718
Runtime error: Unsupported type: list


### Discussion
- Dynamic typing does not mean "accept everything blindly."
- Good Python code clearly defines what kinds of values are acceptable.
- A flexible interface can still be disciplined and explicit.

---

## Problem 9 — Why exact type checking can be too restrictive

**Task:** Show how exact type equality can reject valid objects from subclasses.

**Best practice:** Prefer `isinstance()` when subtype support is desirable.

In [9]:
# Solution 9
class MyInt(int):
    pass

def process_number_bad(x):
    if type(x) is not int:
        raise TypeError("Expected exactly int")
    return x + 1

def process_number_better(x):
    if not isinstance(x, int):
        raise TypeError("Expected an int-compatible value")
    return x + 1

n = MyInt(10)

try:
    print(process_number_bad(n))
except TypeError as ex:
    print("Bad version error:", ex)

print("Better version:", process_number_better(n))

Bad version error: Expected exactly int
Better version: 11


### Discussion
- `MyInt` is a subclass of `int`.
- Exact type checks reject it unnecessarily.
- `isinstance()` is more aligned with Python's polymorphic model.

---

## Problem 10 — Synthesize the concept

**Task:** Summarize what dynamic typing really means in Python.

### Solution summary
Dynamic typing in Python means:

- Names are not permanently typed.
- Objects have types.
- A name may be rebound to objects of different types over time.
- Operations succeed or fail depending on the behavior of the object currently referenced.
- Type hints can document intent, but they do not automatically enforce runtime constraints.

## Final takeaways
1. The type belongs to the object, not the variable name.
2. Rebinding a name is different from mutating an object.
3. Prefer behavior-oriented design when it improves flexibility.
4. Use `isinstance()` more often than exact `type(...) is ...` checks.
5. Dynamic typing is powerful, but clear interfaces and validation still matter.